In [ ]:
"""
SIMULACION MONTECARLO - POLITICA (Q,r) - VERSION FINAL PORTAFOLIO
=====================================================================
Flujo: corrida piloto -> N optimo -> busqueda completa grueso->fino
con ese N optimo -> exporta tabla completa de combinaciones evaluadas
por producto + tabla resumen final.
"""

import numpy as np
import pandas as pd
from scipy.stats import norm

SEMILLA = 42
np.random.seed(SEMILLA)

# ============================================================
# 1. DATOS POR PRODUCTO
# ============================================================
PRODUCTOS = {
    'P001': {
        'nombre': 'Laptop ProBook',
        'forecast_mensual': 22,
        'rmse_mensual': 2.10,
        'costo_unitario': 450000,
        'inventario_inicial': 0
    },
    'P002': {
        'nombre': 'Monitor 24"',
        'forecast_mensual': 20,
        'rmse_mensual': 1.32,
        'costo_unitario': 180000,
        'inventario_inicial': 0
    },
    'P004': {
        'nombre': 'Mouse Optico',
        'forecast_mensual': 35,
        'rmse_mensual': 1.73,
        'costo_unitario': 18000,
        'inventario_inicial': 0
    }
}

# ============================================================
# 2. SUPUESTOS OPERATIVOS
# ============================================================
LEAD_TIME_MESES = 15 / 30
TASA_COSTO_CAPITAL = 0.20
COSTO_ORDENAR = 50000
TASA_COSTO_QUIEBRE = 0.30
MESES_SIMULACION = 12
Z_95_SERVICIO = norm.ppf(0.95)

N_PILOTO = 200
Z_ALPHA_2 = 1.96
MARGEN_ERROR_PORCENTAJE = 0.01
NIVEL_SERVICIO_MIN = 0.95


# ============================================================
# 3. MOTOR DE SIMULACION MENSUAL
# ============================================================
def simular_politica_mensual(Q, r, forecast, rmse, costo_unitario,
                              inventario_inicial, n_replicas,
                              meses=MESES_SIMULACION, semilla=SEMILLA):
    np.random.seed(semilla)  # CRN

    costo_mantener_mensual = (TASA_COSTO_CAPITAL * costo_unitario) / 12
    costo_quiebre_unitario = TASA_COSTO_QUIEBRE * costo_unitario

    inventario = np.full(n_replicas, inventario_inicial, dtype=float)
    pendiente = np.zeros(n_replicas)
    meses_para_llegada = np.zeros(n_replicas)

    costo_total = np.zeros(n_replicas)
    demanda_total = np.zeros(n_replicas)
    quiebre_total = np.zeros(n_replicas)

    for _ in range(meses):
        llegan = (pendiente > 0) & (meses_para_llegada <= 0)
        inventario[llegan] += pendiente[llegan]
        pendiente[llegan] = 0

        en_camino = pendiente > 0
        meses_para_llegada[en_camino] -= 1

        demanda_mes = np.maximum(np.random.normal(forecast, rmse, n_replicas), 0)
        demanda_total += demanda_mes

        quiebre_mes = np.maximum(demanda_mes - inventario, 0)
        inventario = np.maximum(inventario - demanda_mes, 0)
        quiebre_total += quiebre_mes

        generar_pedido = (inventario <= r) & (pendiente == 0)
        pendiente[generar_pedido] = Q
        meses_para_llegada[generar_pedido] = LEAD_TIME_MESES
        costo_total[generar_pedido] += COSTO_ORDENAR

        costo_total += inventario * costo_mantener_mensual
        costo_total += quiebre_mes * costo_quiebre_unitario

    nivel_servicio = 1 - (quiebre_total / demanda_total)

    return dict(
        costo_total=costo_total,
        nivel_servicio=nivel_servicio,
        demanda_total=demanda_total,
        quiebre_total=quiebre_total
    )


def definir_rangos_gruesos(forecast, rmse, costo_unitario, lead_time_meses, z=Z_95_SERVICIO):
    D_anual = forecast * 12
    H_anual = TASA_COSTO_CAPITAL * costo_unitario
    Q_eoq = np.sqrt((2 * D_anual * COSTO_ORDENAR) / H_anual)
    r_centro = (forecast * lead_time_meses) + (z * rmse * np.sqrt(lead_time_meses))
    valores_Q = np.linspace(0.5 * Q_eoq, 8 * Q_eoq, 8).round().astype(int)
    valores_r = np.linspace(max(1, 0.5 * r_centro), 5 * r_centro, 8).round().astype(int)
    return Q_eoq, r_centro, valores_Q, valores_r


def grid_search(valores_Q, valores_r, forecast, rmse, costo_unitario,
                 inventario_inicial, n_replicas, fase, producto_id):
    resultados = []
    for Q in valores_Q:
        for r in valores_r:
            res = simular_politica_mensual(
                Q, r, forecast, rmse, costo_unitario,
                inventario_inicial, n_replicas
            )
            resultados.append({
                'producto_id': producto_id,
                'fase': fase,
                'Q': Q, 'r': r,
                'costo_esperado': round(res['costo_total'].mean()),
                'nivel_servicio': round(res['nivel_servicio'].mean()*100, 1)
            })
    return pd.DataFrame(resultados)


# ============================================================
# 4. FLUJO: piloto -> N optimo -> busqueda completa
# ============================================================
def analizar_producto(producto_id, datos, nivel_servicio_min=NIVEL_SERVICIO_MIN):
    forecast = datos['forecast_mensual']
    rmse = datos['rmse_mensual']
    costo_unitario = datos['costo_unitario']
    inv_inicial = datos['inventario_inicial']

    # Piloto con politica de referencia (EOQ/ROP clasico)
    Q_eoq, r_centro, valores_Q_g, valores_r_g = definir_rangos_gruesos(
        forecast, rmse, costo_unitario, LEAD_TIME_MESES
    )
    Q_referencia = int(round(Q_eoq))
    r_referencia = int(round(r_centro))

    res_piloto = simular_politica_mensual(
        Q_referencia, r_referencia, forecast, rmse,
        costo_unitario, inv_inicial, N_PILOTO
    )
    sigma_costo = res_piloto['costo_total'].std()
    costo_promedio_piloto = res_piloto['costo_total'].mean()
    E = MARGEN_ERROR_PORCENTAJE * costo_promedio_piloto
    N_optimo = int(np.ceil((Z_ALPHA_2 * sigma_costo / E) ** 2))

    # Busqueda GRUESA
    df_grueso = grid_search(valores_Q_g, valores_r_g, forecast, rmse,
                             costo_unitario, inv_inicial, N_optimo,
                             'grueso', producto_id)
    factibles_g = df_grueso[df_grueso['nivel_servicio'] >= nivel_servicio_min*100]
    mejor_g = (factibles_g.loc[factibles_g['costo_esperado'].idxmin()]
               if len(factibles_g) > 0
               else df_grueso.loc[df_grueso['costo_esperado'].idxmin()])

    # Busqueda FINA
    paso_Q = max(1, int(valores_Q_g[1] - valores_Q_g[0]))
    paso_r = max(1, int(valores_r_g[1] - valores_r_g[0]))
    valores_Q_f = np.unique(np.linspace(
        max(1, mejor_g['Q']-paso_Q), mejor_g['Q']+paso_Q, 10).round().astype(int))
    valores_r_f = np.unique(np.linspace(
        max(0, mejor_g['r']-paso_r), mejor_g['r']+paso_r, 10).round().astype(int))

    df_fino = grid_search(valores_Q_f, valores_r_f, forecast, rmse,
                           costo_unitario, inv_inicial, N_optimo,
                           'fino', producto_id)
    factibles_f = df_fino[df_fino['nivel_servicio'] >= nivel_servicio_min*100]
    mejor_f = (factibles_f.loc[factibles_f['costo_esperado'].idxmin()]
               if len(factibles_f) > 0
               else df_fino.loc[df_fino['costo_esperado'].idxmin()])

    df_todas_combinaciones = pd.concat([df_grueso, df_fino], ignore_index=True)

    resumen = {
        'producto_id': producto_id,
        'producto_nombre': datos['nombre'],
        'N_optimo_replicas': N_optimo,
        'Q_optimo': int(mejor_f['Q']),
        'r_optimo': int(mejor_f['r']),
        'costo_esperado_anual': int(mejor_f['costo_esperado']),
        'nivel_servicio_pct': mejor_f['nivel_servicio']
    }

    return df_todas_combinaciones, resumen


# ============================================================
# 5. EJECUCION PARA LOS 3 PRODUCTOS
# ============================================================
tablas_por_producto = {}
resumen_final = []

for producto_id, datos in PRODUCTOS.items():
    df_combinaciones, resumen = analizar_producto(producto_id, datos)
    tablas_por_producto[producto_id] = df_combinaciones
    resumen_final.append(resumen)

    # Exportar tabla individual del producto
    df_combinaciones.to_csv(f'combinaciones_{producto_id}.csv', index=False)

df_resumen = pd.DataFrame(resumen_final)
df_resumen.to_csv('politica_inventario_resumen.csv', index=False)

# Tabla unica con TODAS las combinaciones de los 3 productos juntas
df_todas = pd.concat(tablas_por_producto.values(), ignore_index=True)
df_todas.to_csv('combinaciones_todos_productos.csv', index=False)

# ============================================================
# 6. RESULTADOS PARA EL README
# ============================================================
print("POLITICA (Q,r) OPTIMA POR PRODUCTO")
print(df_resumen.to_string(index=False))

POLITICA (Q,r) OPTIMA POR PRODUCTO
producto_id producto_nombre  N_optimo_replicas  Q_optimo  r_optimo  costo_esperado_anual  nivel_servicio_pct
       P001  Laptop ProBook                 59        47        42               7774628                83.2
       P002     Monitor 24"                 54        44        37               3085243                83.2
       P004    Mouse Optico                 30       106        48                707445                83.1
